# Feinstein (2024) rotator ingestion

Build a reviewable set of new M-dwarf rotators from Feinstein et al. (2024).

This follows the existing `cf_xmatch` preprocessing conventions:

1. load the published Feinstein rotation catalog and retain `Good == 1`;
2. exclude rows flagged as eclipsing binaries;
3. recover Gaia DR3 IDs through Phil's original `dr3_xmatch_from_lcgen` file;
4. remove Gaia IDs already present in `training_stars.csv`;
5. query Gaia DR3 astrometry and photometry;
6. derive extinction using the same Edenhofer/Bayestar prescription;
7. crossmatch to 2MASS and apply the Mann (2019) mass relation;
8. retain the valid Mann range, `4.5 < M_Ks < 10.5`;
9. compute Gossage et al. (2024) turnover times and Rossby numbers;
10. write staged candidates and a separate combined catalog.

The notebook never overwrites `training_stars.csv`.

Expected starting sanity checks:

- 1,848 published rows have `Good == 1`;
- these represent 1,846 unique TIC IDs before the EB cut;
- approximately 1,843 have Gaia DR3 IDs in the supplied LCGen mapping;
- approximately 287 already occur in the current training catalog.

In [13]:
from pathlib import Path
from io import StringIO
import os
import sys
import time

import numpy as np
import pandas as pd
import requests
from scipy.interpolate import interp1d

CF_DATA = Path("cf_data")
WIP_DATA = Path("wip_data")
WIP_DATA.mkdir(exist_ok=True)

TRAIN_PATH = CF_DATA / "training_stars.csv"
FEINSTEIN_URL = (
    "https://raw.githubusercontent.com/afeinstein20/"
    "young-stellar-flares/paper/src/data/combined_prot_flarerate_v6.csv"
)

LCGEN_CANDIDATES = [
    Path("../dr3_xmatch_from_lcgen (1).csv"),
    CF_DATA / "dr3_xmatch_from_lcgen.csv",
]

GAIA_CACHE = WIP_DATA / "feinstein_gaia.csv"
EXTINCTION_CACHE = WIP_DATA / "feinstein_extinction.csv"
TMASS_CACHE = WIP_DATA / "feinstein_2mass.csv"
RAW_CANDIDATES_PATH = WIP_DATA / "feinstein_raw_candidates.csv"
FINAL_CANDIDATES_PATH = CF_DATA / "feinstein_candidates.csv"
COMBINED_PATH = CF_DATA / "training_stars_with_feinstein.csv"

EXCLUDE_ECLIPSING_BINARIES = True
MANN_MKS_MIN = 4.5
MANN_MKS_MAX = 10.5

train = pd.read_csv(TRAIN_PATH, dtype={"source_id": "string"})
train["source_id"] = train["source_id"].str.replace(r"\.0$", "", regex=True)
TRAIN_SCHEMA = train.columns.tolist()

print(f"training_stars: {len(train)} rows")
print(f"training schema: {len(TRAIN_SCHEMA)} columns")

training_stars: 6128 rows
training schema: 35 columns


## Load and quality-filter Feinstein

`Good == 1` is the paper's robust-period flag. Exact duplicate TIC rows are
collapsed after sorting by period uncertainty. Rows marked `EB? == 1` are
excluded by default to remain consistent with the single-star training sample.

In [14]:
feinstein = pd.read_csv(FEINSTEIN_URL, dtype={"Target_ID": "string"})
feinstein["Target_ID"] = (
    feinstein["Target_ID"].str.replace(r"\.0$", "", regex=True)
)

for col in ["Prot", "e_Prot", "Good", "EB?", "age", "Teff", "mass", "e_mass"]:
    feinstein[col] = pd.to_numeric(feinstein[col], errors="coerce")

quality = feinstein[feinstein["Good"].eq(1)].copy()
print(f"Good == 1 rows: {len(quality)}")
print(f"Good == 1 unique TIC IDs: {quality['Target_ID'].nunique()}")
print(f"EB-flagged robust rows: {quality['EB?'].eq(1).sum()}")

if EXCLUDE_ECLIPSING_BINARIES:
    quality = quality[~quality["EB?"].eq(1)].copy()

quality = (
    quality.sort_values(["Target_ID", "e_Prot"], na_position="last")
    .drop_duplicates("Target_ID", keep="first")
    .reset_index(drop=True)
)

assert quality["Prot"].notna().all()
assert quality["Target_ID"].is_unique
print(f"Unique quality rotators after EB cut: {len(quality)}")

Good == 1 rows: 1848
Good == 1 unique TIC IDs: 1846
EB-flagged robust rows: 2
Unique quality rotators after EB cut: 1844


## Recover Gaia DR3 IDs and remove existing stars

Phil's LCGen file is used only as the TIC-to-Gaia mapping and association-age
source. The null Feinstein `Prot` values in that file are replaced by the
published periods loaded above.

In [15]:
lcgen_path = next((p for p in LCGEN_CANDIDATES if p.exists()), None)
if lcgen_path is None:
    raise FileNotFoundError(
        "Could not find Phil's dr3_xmatch_from_lcgen file. "
        f"Checked: {LCGEN_CANDIDATES}"
    )

lcgen = pd.read_csv(
    lcgen_path,
    dtype={"TIC_ID": "string", "GaiaDR3_ID": "string"},
)
lcgen = lcgen[lcgen["ref"].eq("Feinstein2024")].copy()

for col in ["TIC_ID", "GaiaDR3_ID"]:
    lcgen[col] = (
        lcgen[col]
        .replace({"NULL": pd.NA, "nan": pd.NA})
        .str.replace(r"\.0$", "", regex=True)
    )

lcgen["ageMyr"] = pd.to_numeric(lcgen["ageMyr"], errors="coerce")
lcgen["e_ageMyr"] = pd.to_numeric(lcgen["e_ageMyr"], errors="coerce")

mapping = (
    lcgen[["TIC_ID", "GaiaDR3_ID", "ageMyr", "e_ageMyr"]]
    .dropna(subset=["TIC_ID"])
    .sort_values(["TIC_ID", "GaiaDR3_ID"], na_position="last")
    .drop_duplicates("TIC_ID", keep="first")
)

candidates = quality.merge(
    mapping,
    how="left",
    left_on="Target_ID",
    right_on="TIC_ID",
    validate="one_to_one",
)

print(f"Published quality rotators: {len(candidates)}")
print(f"With Gaia DR3 ID: {candidates['GaiaDR3_ID'].notna().sum()}")
print(f"Missing Gaia DR3 ID: {candidates['GaiaDR3_ID'].isna().sum()}")

candidates = candidates.dropna(subset=["GaiaDR3_ID"]).copy()
candidates = (
    candidates.sort_values(["GaiaDR3_ID", "e_Prot"], na_position="last")
    .drop_duplicates("GaiaDR3_ID", keep="first")
)

existing_ids = set(train["source_id"].dropna())
candidates["already_in_training"] = candidates["GaiaDR3_ID"].isin(existing_ids)

print(f"Already in training_stars: {candidates['already_in_training'].sum()}")
candidates = candidates[~candidates["already_in_training"]].copy()
print(f"New Gaia DR3 candidates: {len(candidates)}")

candidates.to_csv(RAW_CANDIDATES_PATH, index=False)
print(f"Saved {RAW_CANDIDATES_PATH}")

Published quality rotators: 1844
With Gaia DR3 ID: 1843
Missing Gaia DR3 ID: 1
Already in training_stars: 287
New Gaia DR3 candidates: 1556
Saved wip_data\feinstein_raw_candidates.csv


## Gaia DR3 crossmatch

The query follows `mocadb_xmatch.ipynb`, with the extra coordinates, proper
motions, parallax uncertainty, and BP/RP magnitudes needed by later stages.
Results are cached so rerunning the notebook does not repeat the TAP query.

In [16]:
from astroquery.gaia import Gaia

def chunks(values, size=400):
    for start in range(0, len(values), size):
        yield values[start:start + size]

def query_gaia(ids):
    ids_text = ",".join(ids)
    core_query = f'''
        SELECT source_id, ra, dec, pmra, pmdec, parallax, parallax_error,
               phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag,
               bp_rp, ruwe
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({ids_text})
    '''
    ap_query = f'''
        SELECT source_id, ebpminrp_gspphot
        FROM gaiadr3.astrophysical_parameters
        WHERE source_id IN ({ids_text})
    '''
    core = Gaia.launch_job(core_query).get_results().to_pandas()
    ap = Gaia.launch_job(ap_query).get_results().to_pandas()
    return core.merge(ap, on="source_id", how="left")

if GAIA_CACHE.exists():
    gaia = pd.read_csv(GAIA_CACHE, dtype={"source_id": "string"})
    print(f"Loaded Gaia cache: {len(gaia)} rows")
else:
    parts = []
    ids = candidates["GaiaDR3_ID"].tolist()
    batches = list(chunks(ids))
    for i, batch in enumerate(batches, start=1):
        print(f"Gaia batch {i}/{len(batches)}")
        parts.append(query_gaia(batch))
    gaia = pd.concat(parts, ignore_index=True)
    gaia["source_id"] = gaia["source_id"].astype("Int64").astype("string")
    gaia.to_csv(GAIA_CACHE, index=False)
    print(f"Saved Gaia cache: {GAIA_CACHE}")

gaia["source_id"] = gaia["source_id"].str.replace(r"\.0$", "", regex=True)
candidates = candidates.merge(
    gaia,
    how="inner",
    left_on="GaiaDR3_ID",
    right_on="source_id",
    validate="one_to_one",
)
candidates["bp_rp_0"] = (
    candidates["bp_rp"] - candidates["ebpminrp_gspphot"].fillna(0)
)
candidates["high_ruwe"] = candidates["ruwe"] >= 1.2

print(f"Gaia-matched candidates: {len(candidates)}")
print(f"RUWE >= 1.2: {candidates['high_ruwe'].sum()}")

Loaded Gaia cache: 1556 rows
Gaia-matched candidates: 1556
RUWE >= 1.2: 422


## Extinction

This is the same prescription as `dereddening.ipynb`: use Edenhofer (2023)
when available, otherwise Bayestar (2019), then convert to `A_Ks` using
Wang & Chen (2019).

The maps are stored in `cf_xmatch/dustmaps_data`. `DUSTMAPS_DATA_DIR` may
override that location.
This stage deliberately fails if the maps are unavailable; it does not silently
assume zero extinction.

In [17]:
from pathlib import Path
import os
import sys

import astropy.units as u
from astropy.coordinates import SkyCoord

# VS Code can retain an older kernel selection. Fall back to the local
# conda environment that contains the compiled healpy dependency.
try:
    import healpy  # noqa: F401
except ModuleNotFoundError:
    env_candidates = [
        Path.cwd() / ".conda-env",
        Path.cwd() / "cf_xmatch" / ".conda-env",
    ]
    dustmaps_env = next((p for p in env_candidates if p.exists()), None)
    if dustmaps_env is None:
        raise RuntimeError("Could not locate the local .conda-env environment.")
    if hasattr(os, "add_dll_directory"):
        _dustmaps_dll_handle = os.add_dll_directory(
            str(dustmaps_env / "Library" / "bin")
        )
    sys.path.insert(0, str(dustmaps_env / "Lib" / "site-packages"))

from dustmaps.config import config
from dustmaps.bayestar import BayestarQuery
from dustmaps.edenhofer2023 import Edenhofer2023Query

if EXTINCTION_CACHE.exists():
    ext = pd.read_csv(EXTINCTION_CACHE, dtype={"source_id": "string"})
    print(f"Loaded extinction cache: {len(ext)} rows")
else:
    default_dust_dirs = [
        Path.cwd() / "dustmaps_data",
        Path.cwd() / "cf_xmatch" / "dustmaps_data",
    ]
    dust_dir = os.environ.get("DUSTMAPS_DATA_DIR")
    if dust_dir is None:
        dust_dir = next((str(p) for p in default_dust_dirs if p.exists()), None)
    if dust_dir is None:
        raise RuntimeError(
            "Set DUSTMAPS_DATA_DIR before running this cell. "
            "See dereddening.ipynb for map setup."
        )
    config["data_dir"] = dust_dir
    edenhofer = Edenhofer2023Query(load_samples=False, integrated=True)
    bayestar = BayestarQuery(max_samples=10)

    rng = np.random.default_rng(42)
    rows = []
    for i, row in candidates.reset_index(drop=True).iterrows():
        if i % 100 == 0:
            print(f"Extinction {i}/{len(candidates)}")

        if not np.isfinite(row["parallax"]) or row["parallax"] <= 0:
            continue

        dist = 1000.0 / row["parallax"]
        if np.isfinite(row["parallax_error"]) and row["parallax_error"] > 0:
            dist_err = 1000.0 * row["parallax_error"] / row["parallax"]**2
            dist_samples = np.clip(rng.normal(dist, dist_err, 10), 10, None)
        else:
            dist_samples = np.array([max(dist, 10)])

        b_samples, e_samples = [], []
        for d in dist_samples:
            coord = SkyCoord(
                row["ra"] * u.deg,
                row["dec"] * u.deg,
                distance=d * u.pc,
                frame="icrs",
            )
            try:
                result, flags = bayestar(coord, mode="samples", return_flags=True)
                if flags[0] & flags[1]:
                    b_samples.extend(np.asarray(result).ravel())
            except Exception:
                pass
            try:
                value = float(edenhofer(coord, mode="mean"))
                if np.isfinite(value):
                    e_samples.append(value)
            except Exception:
                pass

        if e_samples:
            ebv = np.nanmedian(e_samples) * 0.829
            ebv_err = np.nanstd(e_samples) * 0.829
            dustmap = "Edenhofer"
        elif b_samples:
            ebv = np.nanmedian(b_samples) * 0.88
            ebv_err = np.nanstd(b_samples) * 0.88
            dustmap = "Bayestar"
        else:
            ebv = ebv_err = np.nan
            dustmap = "none"

        av = 3.1 * ebv
        av_err = 3.1 * ebv_err
        aks = 0.078 * av
        aks_err = (
            aks * np.sqrt((av_err / av)**2 + (0.004 / 0.078)**2)
            if np.isfinite(av) and av > 0 else 0.0
        )
        rows.append({
            "source_id": row["source_id"],
            "A_Ks": aks,
            "A_Ks_err": aks_err,
            "dustmap_used": dustmap,
        })

    ext = pd.DataFrame(rows)
    ext.to_csv(EXTINCTION_CACHE, index=False)
    print(f"Saved extinction cache: {EXTINCTION_CACHE}")

ext["source_id"] = ext["source_id"].str.replace(r"\.0$", "", regex=True)
candidates = candidates.merge(
    ext[["source_id", "A_Ks", "A_Ks_err", "dustmap_used"]],
    on="source_id",
    how="left",
    validate="one_to_one",
)
print(candidates["dustmap_used"].value_counts(dropna=False))

Loaded extinction cache: 1553 rows
dustmap_used
Edenhofer    1515
none           38
NaN             3
Name: count, dtype: int64


## 2MASS crossmatch

This follows `twomass_xmatch.ipynb`: Gaia's DR3 best-neighbour table supplies
the 2MASS designation, and VizieR `II/246/out` supplies Ks and its uncertainty.

In [18]:
ARI_SYNC = "https://gaia.ari.uni-heidelberg.de/tap/sync"
VIZIER_SYNC = "https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync"

def tap_json(url, query):
    response = requests.post(
        url,
        data={"REQUEST": "doQuery", "LANG": "ADQL", "FORMAT": "json", "QUERY": query},
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    columns = [item["name"] for item in payload["metadata"]]
    return pd.DataFrame(payload["data"], columns=columns)

if TMASS_CACHE.exists():
    tmass = pd.read_csv(TMASS_CACHE, dtype={"source_id": "string"})
    print(f"Loaded 2MASS cache: {len(tmass)} rows")
else:
    designation_parts = []
    ids = candidates["source_id"].tolist()
    for batch in chunks(ids):
        text = ",".join(batch)
        designation_parts.append(tap_json(
            ARI_SYNC,
            f'''
                SELECT source_id, original_ext_source_id AS twomass_id
                FROM gaiadr3.tmass_psc_xsc_best_neighbour
                WHERE source_id IN ({text})
            ''',
        ))
    designations = pd.concat(designation_parts, ignore_index=True)
    designations["source_id"] = (
        designations["source_id"].astype("Int64").astype("string")
    )
    designations["twomass_id"] = designations["twomass_id"].str.strip()

    photometry_parts = []
    names = designations["twomass_id"].dropna().unique().tolist()
    for batch in chunks(names):
        quoted = ",".join(f"'{name}'" for name in batch)
        photometry_parts.append(tap_json(
            VIZIER_SYNC,
            f'''
                SELECT "2MASS" AS twomass_id,
                       "Kmag" AS k_m,
                       "e_Kmag" AS k_cmsig
                FROM "II/246/out"
                WHERE "2MASS" IN ({quoted})
            ''',
        ))
    photometry = pd.concat(photometry_parts, ignore_index=True)
    photometry["twomass_id"] = photometry["twomass_id"].str.strip()
    tmass = designations.merge(photometry, on="twomass_id", how="left")
    tmass.to_csv(TMASS_CACHE, index=False)
    print(f"Saved 2MASS cache: {TMASS_CACHE}")

tmass["source_id"] = tmass["source_id"].str.replace(r"\.0$", "", regex=True)
candidates = candidates.merge(
    tmass[["source_id", "twomass_id", "k_m", "k_cmsig"]],
    on="source_id",
    how="left",
    validate="one_to_one",
)

candidates["k_m_0"] = candidates["k_m"] - candidates["A_Ks"]
candidates["M_Ks"] = (
    candidates["k_m_0"]
    + 5
    - 5 * np.log10(1000.0 / candidates["parallax"])
)
candidates["flag_outside_mann_range"] = ~candidates["M_Ks"].between(
    MANN_MKS_MIN, MANN_MKS_MAX, inclusive="neither"
)

print(f"2MASS matches: {candidates['k_m'].notna().sum()} / {len(candidates)}")
print(
    "In Mann range: "
    f"{(~candidates['flag_outside_mann_range']).sum()} / {len(candidates)}"
)

Loaded 2MASS cache: 1552 rows
2MASS matches: 1552 / 1556
In Mann range: 487 / 1556


## Mann (2019) masses

This uses the same full-posterior calculation as `mass_recompute.ipynb`.

The Mann relation repository is cloned into `cf_xmatch/Mann_2019`. The path
below is resolved from the notebook working directory; `MANN_CODE_DIR` can
still override it when running from another location.

In [19]:
from astropy.io import fits

default_mann_code_dir = Path.cwd() / "Mann_2019"
mann_code_dir = Path(
    os.environ.get("MANN_CODE_DIR", default_mann_code_dir)
).resolve()
if not (mann_code_dir / "mk_mass.py").exists():
    raise FileNotFoundError(
        f"Missing {mann_code_dir / 'mk_mass.py'}. "
        "Set MANN_CODE_DIR to the Mann_2019 code directory."
    )

sys.path.insert(0, str(mann_code_dir))
import mk_mass

resource_dir = Path(mk_mass.__file__).resolve().parent / "resources"
posterior_path = resource_dir / "Mk-M_7_trim.fits"
post_data = fits.getdata(posterior_path)

def mass_percentiles(row):
    if row["flag_outside_mann_range"] or not np.isfinite(row["k_m_0"]):
        return np.nan, np.nan, np.nan

    dist = 1000.0 / row["parallax"]
    edist = 1000.0 * row["parallax_error"] / row["parallax"]**2
    phot_err = row["k_cmsig"] if np.isfinite(row["k_cmsig"]) else 0.0
    ext_err = row["A_Ks_err"] if np.isfinite(row["A_Ks_err"]) else 0.0
    ek = np.hypot(phot_err, ext_err)

    samples = np.asarray(mk_mass.posterior(
        K=row["k_m_0"],
        dist=dist,
        ek=ek,
        edist=edist,
        oned=False,
        silent=True,
        post=post_data,
    ))
    samples = samples[np.isfinite(samples)]
    median, p16, p84 = np.percentile(samples, [50, 16, 84])
    return median, median - p16, p84 - median

results = candidates.apply(mass_percentiles, axis=1)
candidates["mass_msun"] = [result[0] for result in results]
candidates["mass_msun_err_lo"] = [result[1] for result in results]
candidates["mass_msun_err_hi"] = [result[2] for result in results]

print(f"Valid masses: {candidates['mass_msun'].notna().sum()} / {len(candidates)}")

Valid masses: 487 / 1556


## Training schema and Rossby numbers

Age and symmetric age uncertainty come from the Feinstein association labels
in Phil's LCGen mapping. Turnover times use the same Gossage et al. (2024)
interpolation as `rossby_number.ipynb`.

In [20]:
gossage_mass = np.array(
    [0.18, 0.22, 0.30, 0.38, 0.43, 0.53, 0.60, 0.67,
     0.73, 0.81, 0.89, 0.96, 1.01, 1.09]
)
gossage_tau = np.array(
    [214.59, 203.91, 239.77, 135.36, 81.54, 64.29, 43.61,
     36.77, 32.49, 23.95, 19.49, 14.58, 10.96, 7.46]
)
gossage_tau_err = np.array(
    [30.90, 38.73, 49.66, 10.04, 5.96, 4.71, 2.50,
     1.79, 1.64, 1.26, 0.73, 0.53, 0.41, 0.42]
)

tau_interp = interp1d(
    gossage_mass, gossage_tau, bounds_error=False,
    fill_value=(gossage_tau[0], gossage_tau[-1]),
)
tau_err_interp = interp1d(
    gossage_mass, gossage_tau_err, bounds_error=False,
    fill_value=(gossage_tau_err[0], gossage_tau_err[-1]),
)

out = pd.DataFrame({
    "source_paper": "Feinstein2024",
    "star_name": "TIC " + candidates["Target_ID"],
    "source_id": candidates["source_id"],
    "ra": candidates["ra"],
    "dec": candidates["dec"],
    "pmra": candidates["pmra"],
    "pmdec": candidates["pmdec"],
    "prot_days": candidates["Prot"],
    "age_gyr": candidates["ageMyr"] / 1000.0,
    "age_err_lo_gyr": candidates["e_ageMyr"] / 1000.0,
    "age_err_hi_gyr": candidates["e_ageMyr"] / 1000.0,
    "age_method": "association",
    "parallax": candidates["parallax"],
    "parallax_error": candidates["parallax_error"],
    "bp_rp": candidates["bp_rp"],
    "bp_rp_0": candidates["bp_rp_0"],
    "ebpminrp_gspphot": candidates["ebpminrp_gspphot"],
    "ruwe": candidates["ruwe"],
    "high_ruwe": candidates["high_ruwe"],
    "phot_g_mean_mag": candidates["phot_g_mean_mag"],
    "binary_type": "none",
    "twomass_id": candidates["twomass_id"],
    "k_m": candidates["k_m"],
    "k_cmsig": candidates["k_cmsig"],
    "A_Ks": candidates["A_Ks"],
    "A_Ks_err": candidates["A_Ks_err"],
    "k_m_0": candidates["k_m_0"],
    "M_Ks": candidates["M_Ks"],
    "flag_outside_mann_range": candidates["flag_outside_mann_range"],
    "mass_msun": candidates["mass_msun"],
    "mass_msun_err_lo": candidates["mass_msun_err_lo"],
    "mass_msun_err_hi": candidates["mass_msun_err_hi"],
})

out = out[out["mass_msun"].notna()].copy()
out["tau_ce_days"] = tau_interp(out["mass_msun"])
out["tau_ce_err_days"] = tau_err_interp(out["mass_msun"])
out["rossby_number"] = out["prot_days"] / out["tau_ce_days"]

out = out[TRAIN_SCHEMA]
assert out["source_id"].is_unique
assert not set(out["source_id"]) & set(train["source_id"])
assert out["prot_days"].notna().all()
assert out["mass_msun"].notna().all()

print(f"Final Feinstein candidates: {len(out)}")
print(out[["prot_days", "age_gyr", "mass_msun", "ruwe"]].describe())

Final Feinstein candidates: 487
        prot_days     age_gyr   mass_msun        ruwe
count  487.000000  487.000000  487.000000  487.000000
mean     3.006285    0.085963    0.540795    1.373466
std      2.618234    0.064402    0.100070    1.388822
min      0.158956    0.004500    0.236855    0.840409
25%      1.065800    0.015000    0.479030    1.026207
50%      2.031462    0.079000    0.553927    1.078366
75%      4.404384    0.127400    0.622951    1.165446
max     16.898852    0.250000    0.673638   17.412214


## Save staged and combined catalogs

Inspect `feinstein_candidates.csv` before using the combined catalog for
training. The original `training_stars.csv` remains unchanged.

In [21]:
out.to_csv(FINAL_CANDIDATES_PATH, index=False)

combined = pd.concat([train[TRAIN_SCHEMA], out], ignore_index=True)
assert combined["source_id"].is_unique
combined.to_csv(COMBINED_PATH, index=False)

print(f"Saved candidates: {FINAL_CANDIDATES_PATH} ({len(out)} rows)")
print(f"Saved combined:   {COMBINED_PATH} ({len(combined)} rows)")
print("Original training_stars.csv was not modified.")

Saved candidates: cf_data\feinstein_candidates.csv (487 rows)
Saved combined:   cf_data\training_stars_with_feinstein.csv (6615 rows)
Original training_stars.csv was not modified.
